# Essentials AI · IA générative

**Objectif :** LLM, prompts, agents, évaluation et fiabilité.

Lance ce notebook depuis le dossier `notebooks/`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../labs"))   # brewly, fakeapi
os.chdir("../data")                               # penguins.csv, titanic.csv, tips.csv

---
## Le paysage de l'IA et des LLM

L'**IA générative** est le dernier étage d'un empilement de notions qu'on confond souvent :

- **Intelligence artificielle** : tout programme qui accomplit une tâche qu'on croyait réservée aux humains.
- **Machine learning** : l'IA qui *apprend* une règle à partir d'exemples au lieu qu'on l'écrive.
- **Deep learning** : le machine learning avec des réseaux de neurones à plusieurs couches.
- **IA générative** : des modèles de deep learning qui *produisent* du contenu (texte, image, son, code).

Un **LLM** (*Large Language Model*) est un modèle génératif de texte : un réseau de neurones de type Transformer, avec des milliards de paramètres, entraîné sur une énorme quantité de texte.

## Comment un LLM écrit : un token à la fois

Un LLM ne « pense » pas une phrase entière. À chaque étape, il calcule une **probabilité pour chaque token possible** (un morceau de mot), en choisit un, l'ajoute au texte, et recommence. On dit qu'il est **autorégressif**.

*Animation interactive « nexttoken » : à voir sur la plateforme web.*

Le modèle produit des **scores bruts** (les *logits*), transformés en probabilités par la fonction **softmax**. La **température** T contrôle à quel point la distribution est piquée :

$$
p_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}
$$

T proche de 0 : le modèle prend presque toujours le token le plus probable (réponses stables). T élevée : il ose des tokens moins probables (plus de variété, plus d'erreurs).

In [ ]:
import numpy as np

logits = np.array([4.0, 3.2, 2.1, 0.5])       # scores pour : "Paris", "Lyon", "Marseille", "Tokyo"
for T in [0.2, 1.0, 2.0]:
    p = np.exp(logits / T) / np.exp(logits / T).sum()
    print(f"T = {T}:", p.round(3))

## Comment on fabrique un assistant

1. **Pré-entraînement** : le modèle apprend à prédire le token suivant sur des milliers de milliards de tokens. Il acquiert la langue et des connaissances, mais il ne sait pas encore « répondre ».
2. **Ajustement par instructions** (*instruction tuning*) : on l'entraîne sur des exemples de questions et de bonnes réponses.
3. **Alignement par préférences** (RLHF, DPO) : des humains, ou d'autres modèles, comparent des réponses ; le modèle apprend à préférer les meilleures. Tu verras RLHF et DPO dans le module Adaptation et évaluation du FullStack.

## Le paysage des modèles

- **Modèles propriétaires**, accessibles par API : la famille Claude d'Anthropic, GPT d'OpenAI, Gemini de Google.
- **Modèles à poids ouverts**, téléchargeables et exécutables chez soi : Llama de Meta, Mistral, Qwen, Gemma… On les trouve sur Hugging Face.
- Les modèles récents sont **multimodaux** : ils lisent des images, parfois du son et de la vidéo.

Le choix dépend de la qualité requise, du coût, de la latence, de la confidentialité des données et des contraintes de déploiement.

## Ton premier appel à un LLM

Sur cette plateforme, le module `llm` appelle un vrai modèle (Claude) depuis Python. `await` attend la réponse, comme dans Jupyter. Si aucun modèle n'est disponible sur ton appareil, une réponse simulée s'affiche.

In [ ]:
import llm
print("LLM disponible :", llm.disponible())
reponse = await llm.complete("En deux phrases, explique à un débutant ce qu'est un token.")
print(reponse)

> **À retenir** : un LLM prédit le token suivant à partir de probabilités. Il n'a pas accès à une base de faits : il peut produire une réponse fausse mais très bien formulée (une *hallucination*). Tout le travail d'ingénierie consiste à l'encadrer.

### Exercice 1 · Softmax avec température

Écris `softmax(logits, temperature=1.0)` qui renvoie un tableau NumPy de probabilités (positives, de somme 1). Pour la stabilité numérique, soustrais le maximum des logits avant l'exponentielle.

<details><summary>Indice</summary>

Divise par la température, soustrais z.max(), applique np.exp, puis divise par la somme.
</details>

In [ ]:
import numpy as np

def softmax(logits, temperature=1.0):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
p = softmax([2.0, 1.0, 0.1])
assert isinstance(p, np.ndarray), "softmax doit renvoyer un tableau NumPy."
assert np.isclose(p.sum(), 1) and (p > 0).all(), "Les probabilités doivent être positives et de somme 1."
assert np.allclose(p, [0.659, 0.242, 0.099], atol=1e-3), f"softmax([2, 1, 0.1]) vaut {p.round(3)}."
assert softmax([2.0, 1.0, 0.1], temperature=0.1)[0] > 0.99, "Avec une température de 0.1, le premier token doit dépasser 99 %."
assert np.isfinite(softmax([1000.0, 999.0])).all(), "Avec des logits très grands, pense à soustraire le maximum."
print("Bravo, tout est correct.")

### Exercice 2 · Un mini-modèle de langue

`modele` associe à chaque mot les probabilités du mot suivant. Écris `generer(modele, debut, n)` qui part du mot `debut` et ajoute à chaque étape le mot suivant **le plus probable** (décodage glouton), `n` fois, ou s'arrête si le mot n'a pas de suite. Renvoie la phrase (mots séparés par des espaces).

<details><summary>Indice</summary>

Garde une liste de mots. À chaque tour : max(suites, key=suites.get) donne le mot le plus probable.
</details>

In [ ]:
modele = {
    "le": {"chat": 0.5, "chien": 0.3, "modèle": 0.2},
    "chat": {"dort": 0.6, "mange": 0.4},
    "dort": {"sur": 0.7, "bien": 0.3},
    "sur": {"le": 1.0},
    "chien": {"court": 1.0},
}

def generer(modele, debut, n):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert generer(modele, "le", 6) == "le chat dort sur le chat dort", f"generer(modele, 'le', 6) renvoie {generer(modele, 'le', 6)!r}."
assert generer(modele, "chien", 5) == "chien court", "Après « court », il n'y a plus de suite : il faut s'arrêter."
assert generer(modele, "le", 0) == "le", "Avec n = 0, on renvoie seulement le mot de départ."
print("Bravo, tout est correct.")

### Exercice 3 · Échantillonner avec top-k

Le décodage glouton tourne en boucle (« le chat dort sur le chat… »). Écris `echantillonner(probas, k, rng)` qui garde les `k` tokens les plus probables (un dictionnaire token → probabilité), **renormalise** leurs probabilités pour que leur somme fasse 1, puis tire un token au hasard avec `rng.choice`.

<details><summary>Indice</summary>

Trie les tokens par probabilité décroissante, garde les k premiers, divise leurs probabilités par leur somme, puis rng.choice(top, p=p).
</details>

In [ ]:
import numpy as np

def echantillonner(probas, k, rng):
    pass

rng = np.random.default_rng(0)

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
import numpy as np
from collections import Counter
rng = np.random.default_rng(1)
tirages = Counter(echantillonner({"a": 0.5, "b": 0.3, "c": 0.15, "d": 0.05}, 2, rng) for _ in range(4000))
assert set(tirages) <= {"a", "b"}, f"Avec k = 2, seuls a et b peuvent sortir ; tu obtiens {set(tirages)}."
assert abs(tirages["a"] / 4000 - 0.625) < 0.03, f"Après renormalisation, a devrait sortir environ 62,5 % du temps ; tu obtiens {tirages['a'] / 40:.1f} %."
assert echantillonner({"x": 0.9, "y": 0.1}, 1, rng) == "x", "Avec k = 1, c'est le décodage glouton."
print("Bravo, tout est correct.")

---
## Le prompt comme interface

Avec un LLM, le **prompt** est l'interface de programmation. Un bon prompt ressemble au brief qu'on donnerait à un collègue brillant qui ne connaît rien au contexte.

## L'anatomie d'un bon prompt

1. **Le rôle et le contexte** : qui est le modèle, pour qui il travaille, pourquoi.
2. **La tâche**, formulée de façon précise et vérifiable.
3. **Les contraintes** : longueur, ton, ce qu'il ne faut pas faire.
4. **Le format de sortie** : liste, tableau, JSON avec des clés précises.
5. **Des exemples** (*few-shot*) quand le format ou le style est subtil.
6. **Les données**, clairement délimitées (balises `<texte>…</texte>`, triples guillemets).

Compare un prompt vague et un prompt précis :

In [ ]:
import llm

vague = "Résume cet avis client : Livraison en 5 jours au lieu de 2, colis abîmé, mais le service client a remboursé les frais de port en une heure."
precis = """Tu analyses des avis clients pour une boutique en ligne.
Lis l'avis entre les balises <avis>.
Réponds uniquement avec un objet JSON contenant :
- "sentiment" : "positif", "negatif" ou "mitige"
- "problemes" : une liste de problèmes courts (3 mots maximum chacun)
- "resume" : une phrase de 15 mots maximum

<avis>Livraison en 5 jours au lieu de 2, colis abîmé, mais le service client a remboursé les frais de port en une heure.</avis>"""

print(await llm.complete(vague))
print("-" * 40)
print(await llm.complete(precis))

## Les techniques qui marchent

- **Few-shot** : montrer deux ou trois exemples d'entrée et de sortie attendue. Très efficace pour imposer un format ou un style.
- **Laisser réfléchir** : demander au modèle de raisonner étape par étape avant de conclure améliore les tâches de logique et de calcul. On peut lui demander de mettre son raisonnement entre balises, puis ne garder que la conclusion.
- **Sortie structurée** : demander du JSON, puis **valider** le résultat en Python. Les API récentes proposent aussi des modes de sortie structurée, qui garantissent un JSON valide.
- **Découper** : plusieurs prompts simples enchaînés valent mieux qu'un prompt géant (*prompt chaining*, leçon sur les workflows).

## Toujours valider la sortie

Un LLM peut répondre avec du texte autour du JSON, entourer le JSON de balises de code, oublier une clé. Ton code doit s'en protéger.

In [ ]:
import json, re

def extraire_json(texte):
    bloc = re.search(r"\{.*\}", texte, re.S)          # du premier { au dernier }
    if not bloc:
        raise ValueError("Aucun JSON dans la réponse")
    return json.loads(bloc.group(0))

fence = "`" * 3                                     # trois accents graves, comme dans une vraie réponse
reponse = f'Voici le résultat :\n{fence}json\n{{"sentiment": "mitige", "problemes": ["retard", "colis abîmé"]}}\n{fence}'
extraire_json(reponse)

## L'injection de prompt

Si ton prompt contient un texte écrit par un utilisateur, ce texte peut contenir des instructions (« ignore les consignes précédentes et… »). Délimite clairement les données, rappelle au modèle de les traiter comme des données, et ne donne jamais à un LLM plus de droits que nécessaire. Le module LLMOps du Lead y consacre un chapitre entier (garde-fous).

> **À retenir** : sois précis, structure, donne des exemples, impose un format, et valide toujours la sortie avant de t'en servir.

### Exercice 1 · Un template de prompt few-shot

Écris `construire_prompt(texte, exemples)` qui renvoie un prompt de classification de tickets de support. Le prompt doit contenir, dans cet ordre : la consigne exacte `Classe le ticket dans une catégorie : facturation, technique ou compte. Réponds par un seul mot.` ; puis chaque exemple sous la forme `Ticket : ...` suivi à la ligne de `Catégorie : ...` ; puis `Ticket : <texte>` et une dernière ligne `Catégorie :`.

<details><summary>Indice</summary>

Construis une liste de lignes, puis "\n".join(lignes). Termine par "Catégorie :" pour que le modèle complète.
</details>

In [ ]:
exemples = [("Ma carte a été débitée deux fois", "facturation"), ("L'appli plante au démarrage", "technique")]

def construire_prompt(texte, exemples):
    pass

print(construire_prompt("Je n'arrive plus à me connecter", exemples))

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
p = construire_prompt("Je n'arrive plus à me connecter", exemples)
assert isinstance(p, str), "construire_prompt doit renvoyer un texte."
assert p.startswith("Classe le ticket dans une catégorie : facturation, technique ou compte."), "Le prompt doit commencer par la consigne exacte."
assert "Ticket : Ma carte a été débitée deux fois\nCatégorie : facturation" in p, "Chaque exemple doit être « Ticket : ... » puis, à la ligne, « Catégorie : ... »."
assert p.rstrip().endswith("Ticket : Je n'arrive plus à me connecter\nCatégorie :"), "Le prompt doit finir par le ticket à classer puis « Catégorie : »."
assert p.index("débitée") < p.index("plante") < p.index("connecter"), "Les exemples viennent avant le ticket à classer, dans l'ordre."
print("Bravo, tout est correct.")

### Exercice 2 · Valider une sortie structurée

Écris `analyser(reponse)` qui extrait le JSON d'une réponse de LLM et vérifie qu'il contient `sentiment` (une valeur parmi `positif`, `negatif`, `mitige`) et `problemes` (une liste). Elle renvoie le dictionnaire, ou lève une `ValueError` avec un message clair dans tous les autres cas (pas de JSON, JSON invalide, clé manquante, valeur interdite).

<details><summary>Indice</summary>

re.search(r"\{.*\}", texte, re.S) trouve le JSON. Attrape json.JSONDecodeError pour la relancer en ValueError.
</details>

In [ ]:
import json, re

def analyser(reponse):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
ok = analyser('Voici :\n```json\n{"sentiment": "negatif", "problemes": ["retard", "casse"]}\n```')
assert ok == {"sentiment": "negatif", "problemes": ["retard", "casse"]}, f"analyser renvoie {ok}."
for mauvais in ["Pas de JSON ici", '{"sentiment": "negatif", "problemes": ["x"', '{"problemes": []}', '{"sentiment": "joyeux", "problemes": []}', '{"sentiment": "positif", "problemes": "aucun"}']:
    try:
        analyser(mauvais)
        assert False, f"analyser({mauvais!r}) aurait dû lever une ValueError."
    except ValueError:
        pass
print("Bravo, tout est correct.")

---
## Ce qu'est vraiment un agent

Un **agent** est un LLM qui **décide lui-même** des étapes à suivre, en appelant des **outils**, dans une **boucle**, jusqu'à atteindre son objectif. C'est l'idée la plus importante de l'IA appliquée actuelle, et un fil rouge de l'Essentials comme du FullStack (LangGraph, MCP).

*Animation interactive « agent » : à voir sur la plateforme web.*

## Les outils

Un **outil** est une fonction de ton code que le modèle peut demander d'appeler. On le décrit au modèle avec un nom, une description et un **schéma JSON** des paramètres :

```json
{
  "name": "meteo",
  "description": "Donne la météo actuelle d'une ville. À utiliser dès qu'on parle du temps qu'il fait.",
  "input_schema": {
    "type": "object",
    "properties": {"ville": {"type": "string", "description": "Nom de la ville, par exemple Lyon"}},
    "required": ["ville"]
  }
}
```

Le modèle n'exécute rien lui-même : il **répond** « appelle `meteo` avec `{"ville": "Lyon"}` ». C'est **ton code** qui exécute la fonction, puis renvoie le résultat au modèle, qui continue.

## La boucle d'un agent

1. On envoie au modèle l'objectif, l'historique et la liste des outils.
2. Le modèle répond soit par un **appel d'outil**, soit par une **réponse finale**.
3. Si c'est un appel d'outil, on l'exécute et on ajoute le résultat (l'**observation**) à l'historique. Retour à l'étape 1.
4. On s'arrête sur la réponse finale, ou après un **nombre maximal d'étapes** pour éviter les boucles infinies.

In [ ]:
import json

def calculatrice(expression):
    autorises = set("0123456789+-*/(). ")
    if not set(expression) <= autorises:
        return "Erreur : expression refusée"
    return str(eval(expression))

OUTILS = {"calculatrice": calculatrice}

def modele_simule(historique):
    """Un faux LLM qui sait utiliser la calculatrice, pour comprendre la boucle."""
    dernier = historique[-1]
    if dernier["role"] == "user":
        return {"type": "outil", "nom": "calculatrice", "args": {"expression": "450 / 12"}}
    return {"type": "final", "texte": f"Cela fait {dernier['content']} heures par semaine."}

historique = [{"role": "user", "content": "450 heures en 12 semaines, ça fait combien par semaine ?"}]
for etape in range(5):
    action = modele_simule(historique)
    print(f"Étape {etape + 1} :", action)
    if action["type"] == "final":
        break
    resultat = OUTILS[action["nom"]](**action["args"])
    historique.append({"role": "outil", "content": resultat})

## Avec un vrai LLM

On peut faire la même chose avec un vrai modèle, en lui demandant de répondre en JSON. Les API proposent un mécanisme natif d'appel d'outils (*tool use* ou *function calling*), plus fiable : c'est lui que tu utiliseras en formation.

In [ ]:
import llm, json, re

consigne = """Tu es un agent. Outils disponibles :
- calculatrice(expression) : calcule une expression arithmétique.
Réponds UNIQUEMENT en JSON, soit {"type": "outil", "nom": "calculatrice", "args": {"expression": "..."}},
soit {"type": "final", "texte": "..."} quand tu as la réponse."""

historique = [{"role": "user", "content": "Une formation de 450 heures sur 60 jours ouvrés : combien d'heures par jour ?"}]
for etape in range(4):
    texte = await llm.chat(historique, system=consigne)
    print(f"Étape {etape + 1} :", texte)
    try:
        action = json.loads(re.search(r"\{.*\}", texte, re.S).group(0))
    except Exception:
        print("Réponse non exploitable, on s'arrête.")
        break
    if action.get("type") != "outil":
        break
    observation = calculatrice(action["args"]["expression"])
    historique += [{"role": "assistant", "content": texte}, {"role": "user", "content": f"Résultat de l'outil : {observation}"}]

## Agent ou workflow ?

Un **workflow** enchaîne des étapes **fixées à l'avance** par le développeur ; un **agent** choisit ses étapes. Les agents sont plus flexibles mais plus chers, plus lents et plus difficiles à fiabiliser. Règle d'or : commence par la solution la plus simple qui marche, et n'ajoute de l'autonomie que si elle apporte un vrai gain.

> **À retenir** : agent = LLM + outils + boucle + condition d'arrêt. Valide les arguments des outils, limite le nombre d'étapes, et demande une confirmation humaine avant toute action irréversible.

### Exercice 1 · Exécuter un appel d'outil sans planter

Écris `executer(appel, outils)` : `appel` est un dictionnaire `{"nom": ..., "args": {...}}` et `outils` un dictionnaire nom → fonction. Elle renvoie `{"ok": True, "resultat": ...}` si tout va bien, et `{"ok": False, "erreur": "..."}` si l'outil est inconnu ou si la fonction lève une exception (arguments manquants, erreur interne). Un agent ne doit jamais planter : l'erreur est renvoyée au modèle, qui peut se corriger.

<details><summary>Indice</summary>

Teste d'abord si le nom existe dans outils, puis entoure l'appel outils[nom](**args) d'un try / except Exception.
</details>

In [ ]:
def meteo(ville):
    donnees = {"Paris": "12 °C, nuageux", "Lyon": "15 °C, soleil"}
    if ville not in donnees:
        raise KeyError(f"Ville inconnue : {ville}")
    return donnees[ville]

OUTILS = {"meteo": meteo}

def executer(appel, outils):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert executer({"nom": "meteo", "args": {"ville": "Lyon"}}, OUTILS) == {"ok": True, "resultat": "15 °C, soleil"}, "L'appel valide doit renvoyer ok et le résultat."
for appel in [{"nom": "bourse", "args": {}}, {"nom": "meteo", "args": {"ville": "Brest"}}, {"nom": "meteo", "args": {}}, {"nom": "meteo", "args": {"pays": "France"}}]:
    r = executer(appel, OUTILS)
    assert isinstance(r, dict) and r.get("ok") is False and r.get("erreur"), f"Pour {appel}, il faut renvoyer ok False et un message d'erreur ; tu renvoies {r}."
print("Bravo, tout est correct.")

### Exercice 2 · Ta propre boucle d'agent

Écris `agent(question, modele, outils, max_etapes=5)`. L'historique commence par `{"role": "user", "content": question}`. À chaque étape : `action = modele(historique)` ; si `action["type"] == "final"`, renvoie `action["texte"]` ; sinon exécute l'outil (avec ta fonction `executer`, recopiée) et ajoute `{"role": "outil", "content": str(résultat_de_executer)}` à l'historique. Si `max_etapes` est atteint, renvoie exactement `Abandon : trop d'étapes`.

<details><summary>Indice</summary>

Une boucle for sur range(max_etapes). Le return dans la boucle gère la réponse finale ; le return après la boucle gère l'abandon.
</details>

In [ ]:
def executer(appel, outils):
    pass   # recopie ta fonction de l'exercice précédent

def agent(question, modele, outils, max_etapes=5):
    pass

# Un faux modèle pour tester : il demande la météo de deux villes puis compare
def modele_test(historique):
    n_obs = sum(1 for m in historique if m["role"] == "outil")
    if n_obs == 0:
        return {"type": "outil", "nom": "meteo", "args": {"ville": "Paris"}}
    if n_obs == 1:
        return {"type": "outil", "nom": "meteo", "args": {"ville": "Lyon"}}
    return {"type": "final", "texte": "Il fait plus chaud à Lyon."}

OUTILS = {"meteo": lambda ville: {"Paris": 12, "Lyon": 15}[ville]}

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
vus = []
def espion(h):
    vus.append([dict(m) for m in h])
    return modele_test(h)
assert agent("Où fait-il le plus chaud ?", espion, OUTILS) == "Il fait plus chaud à Lyon.", "L'agent doit renvoyer le texte final du modèle."
assert vus[0] == [{"role": "user", "content": "Où fait-il le plus chaud ?"}], "L'historique doit commencer par la question de l'utilisateur."
assert len(vus[-1]) == 3 and "15" in vus[-1][2]["content"], "Les observations des outils doivent être ajoutées à l'historique."
boucle = lambda h: {"type": "outil", "nom": "meteo", "args": {"ville": "Paris"}}
assert agent("?", boucle, OUTILS, max_etapes=3) == "Abandon : trop d'étapes", "Un modèle qui boucle doit être arrêté après max_etapes."
casse = lambda h: {"type": "final", "texte": h[-1]["content"]} if len(h) > 1 else {"type": "outil", "nom": "inconnu", "args": {}}
assert "Outil inconnu" in agent("?", casse, OUTILS), "Une erreur d'outil doit être renvoyée au modèle dans l'historique, pas faire planter l'agent."
print("Bravo, tout est correct.")

---
## Modèles de conception : outils et workflows

La plupart des systèmes d'IA en production ne sont pas des agents totalement autonomes, mais des **workflows** : des compositions simples et prévisibles d'appels à un LLM. Voici les cinq motifs à connaître, du plus simple au plus autonome.

| Motif | Principe | Exemple |
| --- | --- | --- |
| **Enchaînement** (*prompt chaining*) | La sortie d'un appel devient l'entrée du suivant, avec des contrôles entre les étapes | Rédiger un plan, le valider, puis rédiger l'article |
| **Routage** (*routing*) | Un premier appel classe la demande, puis on l'envoie au traitement spécialisé | Un ticket de facturation part vers un prompt et des outils différents d'un ticket technique |
| **Parallélisation** | Plusieurs appels indépendants en même temps, puis agrégation | Analyser 50 avis en parallèle ; ou voter entre 3 réponses |
| **Orchestrateur et exécutants** | Un LLM découpe la tâche et délègue les sous-tâches | Modifier plusieurs fichiers d'un projet de code |
| **Évaluateur et optimiseur** | Un appel produit, un autre critique, on boucle jusqu'à ce que ce soit bon | Traduire, relire, corriger |

## Le routage, en code

Le routeur peut être un LLM, un petit modèle de classification, ou même des règles simples : le motif est le même.

In [ ]:
def router(message):
    m = message.lower()
    if any(mot in m for mot in ["facture", "paiement", "remboursement", "débité"]):
        return "facturation"
    if any(mot in m for mot in ["bug", "plante", "erreur", "lent"]):
        return "technique"
    return "general"

TRAITEMENTS = {
    "facturation": lambda m: "→ prompt facturation + outil : historique des paiements",
    "technique": lambda m: "→ prompt technique + outil : journal d'erreurs",
    "general": lambda m: "→ prompt généraliste",
}
for msg in ["J'ai été débité deux fois", "L'appli plante au démarrage", "Vous ouvrez le dimanche ?"]:
    route = router(msg)
    print(f"{msg!r:35} {route:12} {TRAITEMENTS[route](msg)}")

## L'enchaînement avec un contrôle, avec un vrai LLM

In [ ]:
import llm

idee = "une application qui aide les étudiants à réviser avec des quiz générés par IA"
accroche = await llm.complete(f"Écris une accroche marketing de 12 mots maximum pour : {idee}. Réponds par l'accroche seule.")
print("Accroche :", accroche)
if len(accroche.split()) > 15:                      # contrôle entre deux étapes
    accroche = await llm.complete(f"Raccourcis à 12 mots maximum : {accroche}")
traduction = await llm.complete(f"Traduis en anglais, réponds par la traduction seule : {accroche}")
print("Traduction :", traduction)

> **À retenir** : choisis le motif le plus simple qui résout ton problème. Un workflow est plus facile à tester, à surveiller et à expliquer qu'un agent. L'autonomie se justifie quand on ne peut pas prévoir les étapes à l'avance.

### Exercice 1 · Évaluateur et optimiseur

Écris `ameliorer(tache, generer, evaluer, seuil=8, max_tours=4)`. Au premier tour, `brouillon = generer(tache, None)` ; à chaque tour, `note, critique = evaluer(brouillon)` ; si `note >= seuil`, renvoie `(brouillon, note, tour)` où `tour` est le numéro du tour (1 pour le premier) ; sinon `brouillon = generer(tache, critique)` et on recommence. Après `max_tours` tours sans atteindre le seuil, renvoie le **meilleur** brouillon vu, avec sa note et `max_tours`.

<details><summary>Indice</summary>

Garde en mémoire le meilleur (brouillon, note). Une boucle for tour in range(1, max_tours + 1).
</details>

In [ ]:
def ameliorer(tache, generer, evaluer, seuil=8, max_tours=4):
    pass

# Des faux modèles pour tester
def generer_test(tache, critique):
    return tache if critique is None else tache + " !"

def evaluer_test(texte):
    note = 5 + texte.count("!")
    return note, "Plus d'enthousiasme."

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert ameliorer("Bonjour", generer_test, evaluer_test, seuil=6) == ("Bonjour !", 6, 2), f"Avec seuil=6 : {ameliorer('Bonjour', generer_test, evaluer_test, seuil=6)}, j'attendais ('Bonjour !', 6, 2)."
g = lambda t, c: t + "!" if c else t
assert ameliorer("x", g, lambda s: (5 + s.count("!"), "+"), seuil=20, max_tours=3) == ("x!", 6, 3), "Sans atteindre le seuil, il faut renvoyer le meilleur brouillon, sa note et max_tours."
n = []
ameliorer("a", lambda t, c: n.append(1) or t, lambda s: (9, ""), seuil=8)
assert len(n) == 1, "Si le premier brouillon suffit, on ne doit pas régénérer."
print("Bravo, tout est correct.")

---
## Tokens, fenêtre de contexte et coûts

Les LLM ne lisent pas des lettres ni des mots, mais des **tokens** : des morceaux de mots issus d'un découpage appris. C'est l'unité de tout : la **fenêtre de contexte**, la **facturation**, la **vitesse**.

## Comment on découpe : le BPE

L'algorithme le plus courant est le **BPE** (*Byte Pair Encoding*). On part des caractères, puis on fusionne, encore et encore, la paire de symboles voisins la plus fréquente dans le corpus. Les mots fréquents deviennent un seul token, les mots rares restent découpés.

*Animation interactive « bpe » : à voir sur la plateforme web.*

In [ ]:
from collections import Counter

corpus = ["l o w", "l o w e r", "n e w e s t", "w i d e s t"]
paires = Counter()
for mot in corpus:
    symboles = mot.split()
    for a, b in zip(symboles, symboles[1:]):
        paires[(a, b)] += 1
paires.most_common(3)

Ordres de grandeur utiles : en anglais, un token fait environ 4 caractères, soit trois quarts de mot. Le français, moins représenté dans les données d'entraînement, demande un peu plus de tokens pour le même texte. Le code et les langues rares en demandent encore plus.

## La fenêtre de contexte

C'est le nombre maximal de tokens que le modèle peut traiter **en une fois** : le prompt système, l'historique de la conversation, les documents ajoutés **et** la réponse. Les modèles récents acceptent de 100 000 à plus d'un million de tokens, mais :

- plus de contexte = plus cher et plus lent ;
- les modèles exploitent moins bien l'information noyée au milieu d'un très long contexte ;
- dans une conversation longue, il faut **résumer** ou **tronquer** l'historique.

## Les coûts

Les API facturent au **million de tokens**, avec des prix différents en entrée et en sortie (la sortie coûte en général plusieurs fois plus cher). Un calcul rapide évite les mauvaises surprises :

In [ ]:
def cout(tokens_entree, tokens_sortie, prix_entree_mtok, prix_sortie_mtok):
    return tokens_entree / 1e6 * prix_entree_mtok + tokens_sortie / 1e6 * prix_sortie_mtok

# 10 000 requêtes par jour, 2 000 tokens en entrée, 300 en sortie, avec des prix fictifs de 3 € et 15 € par million
par_jour = 10_000 * cout(2_000, 300, 3, 15)
print(f"{par_jour:.0f} € par jour, soit environ {par_jour * 30:.0f} € par mois")

## Les paramètres d'échantillonnage

- **Température** : la variété des réponses (leçon précédente). Basse pour l'extraction de données et le code, plus haute pour la création.
- **Top-p** (*nucleus sampling*) : on ne garde que les tokens les plus probables dont les probabilités cumulées atteignent p.
- **max_tokens** : la longueur maximale de la réponse. La génération s'arrête net quand elle est atteinte.

> **À retenir** : raisonne toujours en tokens. Estime le coût et la taille du contexte **avant** de construire, et mesure-les ensuite.

### Exercice 1 · Une étape de BPE

Écris `fusion_bpe(corpus)` : elle trouve la paire de symboles voisins la plus fréquente dans le corpus (une liste de mots, symboles séparés par des espaces), puis renvoie un **nouveau corpus** où cette paire est fusionnée partout en un seul symbole. En cas d'égalité, prends la paire qui apparaît la première dans `Counter.most_common`.

<details><summary>Indice</summary>

Counter.update(zip(s, s[1:])) compte les paires d'un mot. Puis reconstruis chaque mot en fusionnant la paire choisie.
</details>

In [ ]:
from collections import Counter

corpus = ["l o w", "l o w e r", "n e w e s t", "w i d e s t"]

def fusion_bpe(corpus):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
c1 = fusion_bpe(["l o w", "l o w e r", "n e w e s t", "w i d e s t"])
assert c1 == ["lo w", "lo w e r", "n e w e s t", "w i d e s t"], f"Après une fusion : {c1}. La paire la plus fréquente (et la première à égalité) est ('l', 'o')."
c3 = fusion_bpe(fusion_bpe(c1))
assert c3 == ["low", "low e r", "n e w es t", "w i d es t"], f"Après trois fusions : {c3}."
print("Bravo, tout est correct.")

### Exercice 2 · Tenir dans le budget de contexte

Écris `tronquer(messages, budget)` : `messages` est une liste de dictionnaires `{"role", "content"}` dont le premier est le message système. Garde **toujours** le message système, puis le plus grand nombre de messages **les plus récents** possible, sans que le total de tokens dépasse `budget`. Compte les tokens avec la fonction `tokens(texte)` fournie. L'ordre chronologique doit être conservé.

<details><summary>Indice</summary>

Parcours les messages du plus récent au plus ancien (reversed), ajoute tant que le budget le permet, puis remets-les dans l'ordre avec [::-1].
</details>

In [ ]:
def tokens(texte):
    return len(texte) // 4 + 1          # estimation grossière : 4 caractères par token

def tronquer(messages, budget):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
conv = [{"role": "system", "content": "s" * 40}] + [{"role": "user" if i % 2 == 0 else "assistant", "content": f"message {i} " + "x" * 30} for i in range(10)]
r = tronquer(conv, 60)
assert r[0] is conv[0], "Le message système doit toujours être gardé, en premier."
assert sum(tokens(m["content"]) for m in r) <= 60, "Le total dépasse le budget."
assert [m["content"][:10] for m in r[1:]] == ["message 6 ", "message 7 ", "message 8 ", "message 9 "], f"Il faut garder les messages les plus récents dans l'ordre ; tu gardes {[m['content'][:10] for m in r[1:]]}."
assert len(tronquer(conv, 1000)) == 11, "Si tout tient dans le budget, il faut tout garder."
print("Bravo, tout est correct.")

---
## Évaluer un système d'IA

« Ça a l'air de marcher » n'est pas une évaluation. Un système d'IA se mesure, comme n'importe quel logiciel se teste. C'est aussi ce qui distingue un profil junior d'un profil solide en entretien : savoir **prouver** qu'un système marche, et savoir quand il ne marche pas.

## Le jeu d'évaluation

Tout commence par un **jeu d'évaluation** (*golden set*) : des dizaines à des centaines d'entrées représentatives, avec la sortie attendue ou des critères de réussite. Il doit contenir des cas faciles, des cas limites, et des cas pièges.

## Trois familles de méthodes

1. **Vérifications automatiques exactes** : la sortie est-elle un JSON valide ? la catégorie est-elle la bonne ? le chiffre est-il exact ? On calcule exactitude, précision, rappel.
2. **Métriques de similarité** avec une référence : recouvrement de mots (BLEU, ROUGE) ou similarité d'embeddings. Utile pour la traduction et le résumé, mais grossier.
3. **LLM juge** (*LLM-as-a-judge*) : un modèle note la réponse selon une **grille** précise. Rapide et souple, mais il faut vérifier que ses notes concordent avec celles d'humains sur un échantillon.

## L'exemple du triage médical

Un assistant classe des messages de patients en `urgent` ou `non urgent`. Une erreur n'a pas le même coût dans les deux sens : rater une urgence est grave. L'exactitude globale ne suffit pas, il faut regarder le **rappel** sur la classe `urgent`.

In [ ]:
import pandas as pd

jeu = pd.DataFrame({
    "message": ["Douleur thoracique depuis 1 h", "Renouvellement d'ordonnance", "Fièvre à 40 chez mon bébé de 2 mois",
                "Question sur ma facture", "Difficulté à respirer", "Rendez-vous de contrôle", "Mal de tête léger", "Saignement abondant"],
    "attendu": ["urgent", "non urgent", "urgent", "non urgent", "urgent", "non urgent", "non urgent", "urgent"],
})
jeu["predit"] = ["urgent", "non urgent", "non urgent", "non urgent", "urgent", "non urgent", "urgent", "urgent"]
exactitude = (jeu.attendu == jeu.predit).mean()
urgents = jeu[jeu.attendu == "urgent"]
rappel_urgent = (urgents.predit == "urgent").mean()
print(f"Exactitude : {exactitude:.0%}   Rappel sur les urgences : {rappel_urgent:.0%}")
jeu[jeu.attendu != jeu.predit]

75 % d'exactitude, mais une urgence pédiatrique ratée : ce système n'est pas déployable en l'état.

## Un LLM juge

In [ ]:
import llm

grille = """Tu évalues la réponse d'un assistant de support. Note de 1 à 5 :
5 = exacte, complète et concise ; 3 = partiellement utile ; 1 = fausse ou hors sujet.
Réponds sur la dernière ligne au format exact « Note : X/5 »."""
question = "Combien de temps les fichiers restent-ils dans la corbeille avec l'offre gratuite ?"
reponse = "Ils restent 30 jours, puis ils sont supprimés définitivement."
verdict = await llm.complete(f"{grille}\n\nQuestion : {question}\nRéponse évaluée : {reponse}")
print(verdict)

## Les bonnes pratiques

- **Automatise** : l'évaluation tourne à chaque changement de prompt ou de modèle, comme des tests de régression.
- **Répète** : un LLM n'est pas déterministe. Fais plusieurs passages et regarde la variance.
- **Lis les erreurs** une par une : c'est là que tu comprends ce qu'il faut corriger.
- **Mesure aussi** le coût, la latence et le taux de refus, pas seulement la qualité.

> **À retenir** : pas de déploiement sans jeu d'évaluation. Choisis la métrique selon le coût des erreurs, et vérifie ton LLM juge contre des notes humaines.

### Exercice 1 · Un banc d'évaluation

Écris `evaluer(modele, jeu)` : `jeu` est une liste de couples `(entrée, attendu)` et `modele` une fonction. Elle renvoie un dictionnaire avec `exactitude` (proportion de bonnes réponses), `rappel_urgent` (proportion des cas attendus `urgent` prédits `urgent`) et `erreurs` (la liste des entrées mal classées, dans l'ordre).

<details><summary>Indice</summary>

Calcule d'abord la liste des prédictions, puis filtre les cas attendus « urgent » pour le rappel.
</details>

In [ ]:
jeu = [("Douleur thoracique", "urgent"), ("Facture", "non urgent"), ("Bébé fièvre 40", "urgent"),
       ("Ordonnance", "non urgent"), ("Saignement abondant", "urgent")]

def modele_test(texte):
    return "urgent" if any(m in texte for m in ["Douleur", "Saignement"]) else "non urgent"

def evaluer(modele, jeu):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
r = evaluer(modele_test, jeu)
assert abs(r["exactitude"] - 0.8) < 1e-9, f"exactitude vaut {r['exactitude']}, j'attendais 0.8."
assert abs(r["rappel_urgent"] - 2 / 3) < 1e-9, f"rappel_urgent vaut {r['rappel_urgent']}, j'attendais 2/3."
assert r["erreurs"] == ["Bébé fièvre 40"], f"erreurs vaut {r['erreurs']}."
print("Bravo, tout est correct.")

### Exercice 2 · Lire la note d'un LLM juge

Écris `lire_note(verdict)` qui extrait la note d'une réponse de LLM juge contenant `Note : X/5` (X entier de 1 à 5, espaces optionnels, majuscule ou minuscule). S'il y a plusieurs notes, prends la **dernière**. Lève une `ValueError` si aucune note valide n'est trouvée.

<details><summary>Indice</summary>

re.findall(r"note\s*:\s*([1-5])\s*/\s*5", texte, re.I) renvoie toutes les notes ; prends la dernière.
</details>

In [ ]:
import re

def lire_note(verdict):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert lire_note("Bonne réponse.\nNote : 4/5") == 4, "Cas simple : Note : 4/5."
assert lire_note("note:2 / 5") == 2, "Les espaces et la casse peuvent varier."
assert lire_note("Brouillon : Note : 3/5. Après relecture, Note : 5/5") == 5, "S'il y a plusieurs notes, prends la dernière."
for mauvais in ["Très bien !", "Note : 7/5", "Note : 4/10"]:
    try:
        lire_note(mauvais)
        assert False, f"lire_note({mauvais!r}) doit lever une ValueError."
    except ValueError:
        pass
print("Bravo, tout est correct.")